# PPO


In [8]:
!uv pip install gymnasium torch swig "gymnasium[box2d]"

Using Python 3.12.13 environment at: /usr
Resolved 36 packages in 191ms                                        
Prepared 3 packages in 179ms                                             
Installed 3 packages in 23ms                                
 + box2d==2.3.10
 + pygame-ce==2.5.7
 + swig==4.4.1


In [2]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt

class ActorPolicyNetwork(nn.Module):
    def __init__(self,
                 input_state_features=8, 
                 num_actions=4,
                 hidden_features=128):
        
        super(ActorPolicyNetwork, self).__init__()

        self.fc1 = nn.Linear(input_state_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, hidden_features)
        self.fc3 = nn.Linear(hidden_features, num_actions)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        pi = F.softmax(self.fc3(x), dim=-1)
        return pi

class CriticValueNetwork(nn.Module):
    def __init__(self, input_state_features=8, hidden_features=128):
        super(CriticValueNetwork, self).__init__()
        self.fc1 = nn.Linear(input_state_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, hidden_features)
        self.fc3 = nn.Linear(hidden_features, 1)  

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        value = self.fc3(x)
        return value

In [3]:
class Memory:
    def __init__(self):
        self.states = []
        self.actions = []
        self.dones = []
        self.next_states = []
        self.rewards = []
        self.probs = []

    def store(self, state, action, done, next_state, reward, prob):
        self.states.append(state)
        self.actions.append(action)
        self.dones.append(0 if done else 1)
        self.next_states.append(next_state)
        self.rewards.append(reward)
        self.probs.append(prob)

    def reset(self):
        self.states = []
        self.actions = []
        self.dones = []
        self.next_states = []
        self.rewards = []
        self.probs = []
        
    def sample(self):
        return (np.array(self.states, dtype=np.float32),
                np.array(self.actions),
                np.array(self.dones, dtype=np.float32),
                np.array(self.next_states, dtype=np.float32),
                np.array(self.rewards, dtype=np.float32),
                np.array(self.probs, dtype=np.float32))

In [ ]:
class Trainer:
    def __init__(self, 
                 env,
                 num_training_iterations=250,
                 gamma=0.995, 
                 lambda_weight=0.99, 
                 weight_decay=0.001, 
                 clip_eps=0.2, 
                 ppo_epochs=10,
                 mini_batch_size=64,
                 policy_lr=3e-4,
                 value_lr=3e-4,
                 batch_size=30_000,
                 print_freq=10, 
                 running_avg_scores=25,
                 target_rewards=200,
                 device="cuda"):

        self.env = env
        self.iterations = num_training_iterations
        self.gamma = gamma
        self.lmbda = lambda_weight
        self.weight_decay = weight_decay
        self.clip_eps = clip_eps
        self.ppo_epochs = ppo_epochs
        self.mini_batch_size = mini_batch_size
        self.batch_size = batch_size
        self.print_freq = print_freq
        self.target_rewards = target_rewards
        self.running_avg_scores = running_avg_scores
        self.device = device

        self.num_inputs = env.observation_space.shape[0]
        self.num_actions = env.action_space.n
        
        self.policy_network = ActorPolicyNetwork(self.num_inputs, self.num_actions).to(device)
        self.value_network = CriticValueNetwork(self.num_inputs).to(device)

        self.policy_optimizer = torch.optim.AdamW(self.policy_network.parameters(), lr=policy_lr, weight_decay=weight_decay)
        self.value_optimizer = torch.optim.AdamW(self.value_network.parameters(), lr=value_lr, weight_decay=weight_decay)

    def select_action(self, state): 

        state = torch.from_numpy(state).unsqueeze(0).to(device=self.device, dtype=torch.float32)
        probs = self.policy_network(state) # returns as 1x4
        action = torch.multinomial(probs, 1)

        ### Grab the probability of the action we took ###
        prob = probs[0, action.item()].detach() # index in and take index of selected action
        
        return action.item(), prob
    
    def get_ppo_loss(self, states, actions, advantages, old_probs): 

        # calculate the surrogate loss 
        
        # with the new policy network calculate the action probs
        action_probs = self.policy_network(states)

        # get probs of the actions taken
        new_probs = action_probs.gather(1, actions.unsqueeze(1)).squeeze(1)

        ratio = new_probs / (old_probs + 1e-8)

        surrogate_loss = ratio * advantages
        
        # clamp the ratio, so if the new policy is much more likely to take this action, it doesn't rocket the advantage
        ### say the new policy thinks that action 2 is much more likely, 
        clipped_surrogate_loss = torch.clamp(ratio, 1.0 - self.clip_eps, 1.0 + self.clip_eps) * advantages
        policy_loss = -torch.min(surrogate_loss, clipped_surrogate_loss).mean() # choose the smaller of the loss, and then average across the sampled advantages

        return policy_loss

    def update_model(self, batch): 
        states, actions, dones, next_states, rewards, old_probs = batch

        # initialize experience tensors
        rewards = torch.tensor(rewards, dtype=torch.float32, device=self.device)
        dones = torch.tensor(dones, dtype=torch.float32, device=self.device)
        actions = torch.tensor(actions, dtype=torch.long, device=self.device)
        states = torch.tensor(states, dtype=torch.float32, device=self.device)
        old_probs = torch.tensor(old_probs, dtype=torch.float32, device=self.device)
        
        values = self.value_network(states)
        returns = torch.zeros_like(values)
        advantages = torch.zeros_like(values)
        
        episode_ends = [i for i in range(len(dones)) if dones[i] == 0] + [len(dones)] # list of all the done indices like [1, 104, 193, batch_size]
        episode_starts = [0] + [i+1 for i in episode_ends[:-1]]

        ### calculate advantages and rewards per episode 
        for start, end in zip(episode_starts, episode_ends):
            episode_rewards = rewards[start:end]
            episode_dones = dones[start:end]
            episode_values = values[start:end]

            future_rewards = 0
            future_values = 0
            future_advantages = 0
            
            for i in reversed(range(len(episode_rewards))):
                # start from the last experience in the episode
                idx = start + i
                # return is the rt + gamma*rt+1 + gamma^2 * rt+2 + ...
                returns[idx] = episode_rewards[i] + self.gamma * future_rewards * episode_dones[i]
                td_error = episode_rewards[i] + self.gamma * future_values * episode_dones[i] - episode_values[i]
                # advantages is GAE formula, where A_t+mini_batch_size = td_error, and A_t = A_1 + gamma * lambda A_2 + ...
                advantages[idx] = td_error + self.gamma * self.lmbda * future_advantages * episode_dones[i]
                
                # save so you recursively update returns and advantages arrays
                ### returns  and advantages are tensors with the respective returns of each episode in them
                
                # these ones just contain values for this episode - and are used for recursion
                future_rewards = returns[idx]
                future_values = episode_values[i]
                future_advantages = advantages[idx]
    
        advantages = ((advantages - advantages.mean()) / (advantages.std() + 1e-8)).detach()

        dataset_size = len(states)
        indices = np.arange(0, dataset_size, self.mini_batch_size)

        for epoch in tqdm(range(self.ppo_epochs), unit="epoch", leave=False):

            # shuffle random indices to pick out random batches
            np.random.shuffle(indices)

            for start in range(0, dataset_size, self.mini_batch_size):
                
                end = start + self.mini_batch_size

                # what are indices of the experiences from that batch
                sampled_indices = indices[start:end]

                ### Grab data from our memories at those random indices ###
                sampled_states = states[sampled_indices]
                sampled_actions = actions[sampled_indices]
                sampled_advantages = advantages[sampled_indices]
                sampled_returns = returns[sampled_indices]
                sampled_old_probs = old_probs[sampled_indices]

                # update policy network with sampled experiences
                self.policy_optimizer.zero_grad()
                policy_loss = self.get_ppo_loss(sampled_states, sampled_actions, sampled_advantages, sampled_old_probs)
                policy_loss.backward()
                self.policy_optimizer.step()

                # update our critic model
                self.value_optimizer.zero_grad()
                sampled_values = self.value_network(sampled_states)
                # MSE loss
                value_loss = ((sampled_values-sampled_returns)**2).mean()
                value_loss.backward()
                self.value_optimizer.step()

    def train(self):

        memory = Memory()

        log = {"scores": [], "running_avg_scores": []}

        for i in range(self.iterations): # iterations is number of batches to run through

            num_episodes = 0
            total_reward = 0
            num_steps = 0

            # batch size dictates how often we want to update the model
            with tqdm(total=self.batch_size, desc=f"Iter {i+1} Data Collection", unit="step", leave=False) as collector_bar:
                # take X steps across as many episodes as needed to get to batch_size experiences 
                while num_steps <= self.batch_size: 
                    state, _ = self.env.reset()
                    done = False
                    episode_score = 0
                    episode_steps = 0

                    while not done:
                        action, prob = self.select_action(state)
                        next_state, reward, terminal, truncate, _ = self.env.step(action)
                        done = terminal or truncate

                        episode_steps += 1
                        episode_score += reward

                        memory.store(state, action, done, next_state, reward, prob.cpu().detach().numpy()) # last index is old_prob, pi_k

                        # continue from the next state
                        state = next_state
                        collector_bar.update(1)

                    total_reward += episode_score
                    num_steps += episode_steps
                    num_episodes += 1
                    log["scores"].append(episode_score)
                    running_avg_score = np.mean(log["scores"][-self.running_avg_scores:])
                    log["running_avg_scores"].append(running_avg_score)

            total_reward = total_reward / num_episodes
            batch = memory.sample() # sample returns all of the experiences
            self.update_model(batch)
            memory.reset()

            ### continue to update the model until the target_reward is met or max number of iterations is met
            if i % self.print_freq == 0: 
                print(f'Episode {i}, Avg Batch Rewards: {total_reward: .2f}')

            if total_reward > self.target_rewards: 
                print(f'Achieved Reward {total_reward}, Completed Training!')
                break
        
        return self.policy_network, self.value_network, log


In [ ]:
env = gym.make("LunarLander-v3")
print("TRAINING PPO")
trainer = Trainer(env, device="cuda")
policy, value, log = trainer.train()

TRAINING PPO


Episode 0, Avg Batch Rewards: -175.90


Episode 10, Avg Batch Rewards: -164.53


Episode 20, Avg Batch Rewards: -165.69


Episode 30, Avg Batch Rewards: -150.26


...

Episode 70, Avg Batch Rewards: -111.22


Episode 80, Avg Batch Rewards: -187.47


Episode 90, Avg Batch Rewards: -271.62


Episode 100, Avg Batch Rewards: -303.84


Iter 103 Data Collection:  38%|███▊      | 11449/30000 [00:02<00:03, 5226.24step/s]

In [13]:
plt.figure(figsize=(15,5))
plt.plot(log["scores"], label="Scores")
plt.plot(log["running_avg_scores"], label="Moving Avg")
plt.title("Lunar Lander Scores per Game")
plt.xlabel("Game Number")
plt.ylabel("Score")
plt.legend()
plt.show()

NameError: name 'log' is not defined

<Figure size 1500x500 with 0 Axes>